# Lecture 5 - Building LLM applications with LangChain

This notebook accompanies the study guide. Run the cells **in order** in a Python 3.12
kernel. Local examples run without an API key or model charges. Live examples use
OpenRouter only after you change the switches in §2.1. No hosted tracing or deployment
is enabled automatically. The original notebook is left unchanged.

## Lecture map

1. Why use LangChain?
   - 1.1 The programming problem it solves
   - 1.2 LangChain, LangGraph, and LangSmith: their roles
2. Build and understand a first chain
   - 2.1 Project environment, packages, and API keys
   - 2.2 Connect a model and inspect its reply
   - 2.3 Turn a prompt template into model input
   - 2.4 Runnable and LCEL: connect steps and follow the data
3. Control how the chain runs
   - 3.1 One input, many inputs, streaming, and async calls
   - 3.2 Run independent branches in parallel
   - 3.3 Choose a branch using explicit conditions
4. Return data that application code can use
   - 4.1 Define and validate an output with Pydantic
   - 4.2 Compare Pydantic, TypedDict, and JSON Schema
   - 4.3 Allow alternative response structures
   - 4.4 Display a structured response as it arrives
   - 4.5 Guide content with examples; repair validation failures
5. Extend a chain into an agent workflow
   - 5.1 LangGraph and agent loops: state and saved progress
   - 5.2 Middleware: add behavior around agent execution
6. Inspect, test, and deploy the application
   - 6.1 LangSmith: follow a request through its steps
   - 6.2 Test datasets, evaluators, and human feedback
   - 6.3 Prompt versions, production monitoring, and costs
   - 6.4 Deploy an application so other people can use it

## Before the first run

Select a Python 3.12 environment and install the file beside this notebook:

```text
python -m pip install -r lecture5_requirements.txt
```

In Jupyter, `%pip install -r lecture5_requirements.txt` installs into the selected kernel.
Restart that kernel after installation if packages were already imported.
The exact resolved environment is also recorded in `lecture5_tested_environment.txt`.

## 1. Why use LangChain?

### 1.1 The programming problem it solves

Our article assistant needs to build messages, call a model, extract its answer, validate
fields, and sometimes repeat or route work. LangChain gives these operations compatible
Python interfaces. We still write the task rules and decide what counts as a correct answer.

### 1.2 LangChain, LangGraph, and LangSmith: their roles

| Component | What it does here |
|---|---|
| LangChain | Model adapters, prompts, structured responses, reusable execution steps |
| LangGraph | Runs nodes, carries application state, selects routes, saves progress |
| LangSmith | Optional hosted tracing, evaluation, prompt management and deployment |

LangChain and LangGraph are MIT-licensed open-source libraries. LangSmith is a commercial
platform with free allowances and paid plans; its SDK is separate from that platform.
Using the libraries does not require a LangSmith subscription.

### Where these tools sit

```text
Our Python application
  -> LangChain model adapter
  -> OpenRouter gateway (optional access route)
  -> chosen provider's model
  <- generated response

LangGraph organizes our Python workflow.
LangSmith can record selected operations when we explicitly enable it.
```

OpenRouter is a proprietary hosted gateway, not a model and not part of LangChain. It
provides one API/account for multiple providers; model usage and platform fees are separate
from open-source library licensing. Direct provider APIs, another gateway, or a self-managed
gateway are alternatives. We use this combination to practice model-independent composition
with the account already chosen for this course, not because it is the only or universally
best option. Current prices and supported models: [OpenRouter](https://openrouter.ai/models).

## 2. Build and understand a first chain
### 2.1 Project environment, packages, and API keys

Keep `RUN_LIVE = False` for a no-charge run. Change it to `True` for **three core application-level
requests** (raw reply, text chain, Pydantic response), assuming they succeed.
`RUN_EXTRA_LIVE` enables up to **14 additional application-level requests** across the API
demonstrations, including at most two repairs. `RUN_LIVE_AGENT` separately enables an agent with at most
three model calls. Streaming, batching and parallelism do not make calls free.

The key loader reads the user-approved file only when a live switch is enabled. It accepts
a raw key or an `OPENROUTER_API_KEY=...` assignment. It never prints the key. Private
credentials remain outside the notebook. Never share executed cells that expose private
inputs or outputs. Choose a model/provider that supports function calling and the schemas
used below; `MODEL_ID` is configurable, not a claim about the newest or best model.

In [1]:
from pathlib import Path
import os
import sys
from importlib.metadata import version

RUN_LIVE = False
RUN_EXTRA_LIVE = False
RUN_LIVE_AGENT = False
ENABLE_LANGSMITH = False
MODEL_ID = "openai/gpt-4o-mini"
KEY_PATH = Path(
    "/Users/Shared/Repos/goit_generative_and_agentic_ai/openrouter_api.txt")

# Keep this kernel's automatic hosted tracing off, even if a parent environment
# enabled it.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

def load_openrouter_key(path: Path) -> str:
    lines = [line.strip() for line in path.read_text().splitlines()
             if line.strip() and not line.lstrip().startswith("#")]
    assignments = [
        line for line in lines if line.startswith("OPENROUTER_API_KEY=")
    ]
    if assignments:
        value = assignments[0].split("=", 1)[1].strip().strip("\"'")
    elif len(lines) == 1:
        value = lines[0].strip("\"'")
    else:
        raise ValueError(
            "Use one raw key or one OPENROUTER_API_KEY assignment in "
            "the key file.")
    if not value.startswith("sk-or-") or any(ch.isspace() for ch in value):
        raise ValueError(
            "The key file does not contain a recognizable OpenRouter key.")
    return value

if RUN_LIVE or RUN_LIVE_AGENT:
    os.environ["OPENROUTER_API_KEY"] = load_openrouter_key(KEY_PATH)
    print("OpenRouter credential loaded into this kernel; value "
          "is not displayed.")
else:
    print("Local mode: key file not read; model requests disabled.")

print("Python:", sys.version.split()[0])
for package in ("langchain", "langchain-core", "langchain-openrouter",
                "langgraph", "pydantic"):
    print(package, version(package))

Local mode: key file not read; model requests disabled.
Python: 3.12.14
langchain 1.4.6
langchain-core 1.6.12
langchain-openrouter 0.2.9
langgraph 1.2.16
pydantic 2.12.5


### 2.2 Connect a model and inspect its reply

`ChatOpenRouter` is the model adapter. Calling `invoke` sends messages and returns an
`AIMessage`, which contains text and may include usage metadata. `reply.content` reads
the content, not the whole message. We set an output cap and a 60-second timeout.
This adapter expresses its timeout in milliseconds. Request counts here describe workflow
calls; the SDK/provider's transport retries and billed usage are a separate concern.

In [2]:
from langchain_openrouter import ChatOpenRouter

llm = None
if RUN_LIVE or RUN_LIVE_AGENT:
    llm = ChatOpenRouter(
        model=MODEL_ID,
        temperature=0,
        max_tokens=400,
        timeout=60_000,
        max_retries=0,
    )

if RUN_LIVE:
    reply = llm.invoke([("human",
                         "In one sentence, what does a public library do?")])
    print(type(reply).__name__)
    print(reply.content)
    print("Usage metadata:", reply.usage_metadata)
else:
    print(
        "Live model example skipped. Set RUN_LIVE=True, then rerun from §2.1.")

Live model example skipped. Set RUN_LIVE=True, then rerun from §2.1.


### 2.3 Turn a prompt template into model input

The template stores reusable instructions and an `{article}` slot. Formatting inserts the
article locally; no model request happens here. Adjacent Python strings in parentheses join
into one string without a comma. A comma would create separate tuple items.

`ChatPromptTemplate` constructs role-labelled messages. Its plain-text counterpart,
`PromptTemplate`, formats a string. Both are useful when their output matches the next step.

In [3]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

ARTICLE = (
    "The library will open on Sundays from June. "
    "The trial lasts three months."
)
SECOND_ARTICLE = ("The museum closes for repairs in July. It "
                  "reopens in September.")
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Summarize the article in one sentence. Preserve the dates, durations, "
     "and important conditions. Use only the article's facts."),
    ("human", "{article}"),
])
prompt_value = summary_prompt.invoke({"article": ARTICLE})
for message in prompt_value.to_messages():
    print(message.type, "->", message.content)

plain_prompt = PromptTemplate.from_template("Summarize: {article}")
print("Plain formatted string:", plain_prompt.format(article=ARTICLE))

system -> Summarize the article in one sentence. Preserve the dates, durations, and important conditions. Use only the article's facts.
human -> The library will open on Sundays from June. The trial lasts three months.
Plain formatted string: Summarize: The library will open on Sundays from June. The trial lasts three months.


### 2.4 Runnable and LCEL: connect steps and follow the data

A **Runnable** is an object with LangChain's shared execution interface, including
`invoke`. It can represent one step or a whole workflow. A step's output must be suitable
input for the next step. The `|` expression connects compatible Runnables into a sequence;
this composition syntax is called LCEL (LangChain Expression Language).

Our live chain passes `dict -> formatted messages -> AIMessage -> str`.
`StrOutputParser` extracts text. It does not verify facts.

First, a completely local chain demonstrates that Runnables do not have to call an LLM:
the first function strips spaces, the second counts characters. Its result is a measured
string length, not a generated answer. Then we build the actual model chain separately.

In [4]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

local_chain = RunnableLambda(str.strip) | RunnableLambda(len)
assert local_chain.invoke("  library  ") == 7
print("Local chain result:", local_chain.invoke("  library  "))

summary_chain = summary_prompt | llm | StrOutputParser(
) if llm is not None else None
if RUN_LIVE:
    summary = summary_chain.invoke({"article": ARTICLE})
    print(summary)
else:
    print("The text-model chain is defined only in live mode; no substitute "
          "answer is invented.")

Local chain result: 7
The text-model chain is defined only in live mode; no substitute answer is invented.


## 3. Control how the chain runs
### 3.1 One input, many inputs, streaming, and async calls

`invoke` returns one result. `batch` runs a list of inputs and returns results in input order;
it is not automatically a provider's discounted batch API. `stream` yields pieces while a
result arrives. `ainvoke` can pause a Python coroutine during the network wait.

`await` pauses **that coroutine**; it does not create another task. Below, `asyncio.gather`
explicitly schedules two operations, each of which prints before and after its own wait.
The local timer demonstration makes concurrent work observable without a model call.
In a notebook use top-level `await`; a standalone script usually starts with `asyncio.run`.

In [5]:
import asyncio
import time

async def local_wait(label, delay):
    print(label, "started")
    # A non-blocking stand-in for waiting, not an LLM.
    await asyncio.sleep(delay)
    print(label, "finished")
    return label

started = time.perf_counter()
labels = await asyncio.gather(local_wait("A", 0.15), local_wait("B", 0.10))
print("Results in supplied order:", labels)
print("Elapsed seconds:", round(time.perf_counter() - started, 3))
assert labels == ["A", "B"]

async def summarize_live(label, article):
    print(label, "request starting")
    result = await summary_chain.ainvoke({"article": article})
    print(label, "reply received")
    return result

if RUN_LIVE and RUN_EXTRA_LIVE:
    inputs = [{"article": ARTICLE}, {"article": SECOND_ARTICLE}]
    print("Batch:", summary_chain.batch(inputs, config={"max_concurrency": 2}))
    print("Text stream:", end=" ")
    for piece in summary_chain.stream({"article": ARTICLE}):
        print(piece, end="", flush=True)
    print()
    answer_a, answer_b = await asyncio.gather(
        summarize_live("A", ARTICLE), summarize_live("B", SECOND_ARTICLE),
    )
    print(answer_a, answer_b, sep="\n")
else:
    print("Extra live batch/stream/async examples skipped "
          "(5 requests if enabled).")

A started
B started
B finished
A finished
Results in supplied order: ['A', 'B']
Elapsed seconds: 0.152
Extra live batch/stream/async examples skipped (5 requests if enabled).


### 3.2 Run independent branches in parallel

`RunnableParallel` supplies the **same input** to each named branch, runs those branches
concurrently, and returns a dictionary. The local branches below compute different useful
properties of the article. The live branches separately generate a title and summary,
so we can tune their prompts independently. That live version makes two model requests.

In [6]:
from langchain_core.runnables import RunnableParallel

local_parallel = RunnableParallel(
    characters=RunnableLambda(lambda data: len(data["article"])),
    words=RunnableLambda(lambda data: len(data["article"].split())),
)
print(local_parallel.invoke({"article": ARTICLE}))

title_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a short factual title using only the article."),
    ("human", "{article}"),
])
title_chain = title_prompt | llm | StrOutputParser(
) if llm is not None else None
if RUN_LIVE and RUN_EXTRA_LIVE:
    parallel = RunnableParallel(title=title_chain, summary=summary_chain)
    print(parallel.invoke({"article": ARTICLE}))

{'characters': 73, 'words': 13}


### 3.3 Choose a branch using explicit conditions

`RunnableBranch` checks its conditions in order, executes the first matching branch, or
uses its final default. Python chooses here, not the model. Our local example makes the
selection visible without inference. The live variant uses the same rule to select a
title or summary chain. Every non-`title` value selects the default; add input validation
if your interface must reject unknown operation names.

In [7]:
from langchain_core.runnables import RunnableBranch

def wants_title(data):
    return data["output"] == "title"

local_router = RunnableBranch(
    (wants_title, RunnableLambda(lambda data: "Selected title operation")),
    RunnableLambda(lambda data: "Selected summary operation"),
)
assert local_router.invoke({"output": "title"}) == "Selected title operation"
assert local_router.invoke({"output": "summary"
                            }) == "Selected summary operation"
print(local_router.invoke({"output": "title"}))

if RUN_LIVE and RUN_EXTRA_LIVE:
    router = RunnableBranch((wants_title, title_chain), summary_chain)
    print(router.invoke({"article": ARTICLE, "output": "title"}))

Selected title operation


## 4. Return data that application code can use
### 4.1 Define and validate an output with Pydantic

Our application needs a title and summary as separate named fields. A Pydantic model
defines their types and checks. `Field(description=...)` tells the model what to write;
`min_length` and `max_length` impose local character-count checks. A description saying
"one sentence" does not enforce a sentence count or factual accuracy.

`note: str | None = None` allows text or null **and** gives a default for omission.
Without `= None`, a nullable field would still be required in Pydantic v2.
Validation reports a bad field; it does not truncate or rewrite that field.

In [8]:
from pydantic import BaseModel, Field, ValidationError, ConfigDict

class ArticleCard(BaseModel):
    """A factual title and a one-sentence article summary."""
    model_config = ConfigDict(extra="forbid")
    title: str = Field(min_length=1, max_length=60,
                       description="Short factual title")
    summary: str = Field(min_length=1, max_length=180,
                         description="One-sentence summary")
    note: str | None = None

VALID_CARD = {
    "title": "Sunday library trial",
    "summary": "The library will trial Sunday opening from June "
    "for three months.",
}
checked = ArticleCard.model_validate(VALID_CARD)
assert checked.note is None
print(checked.title)
print(checked.model_dump())

try:
    ArticleCard.model_validate({"title": "Sunday opening", "summary": ""})
except ValidationError as error:
    print("Deliberate local failure:",
          error.errors()[0]["loc"],
          error.errors()[0]["type"])

if RUN_LIVE:
    card_chain = summary_prompt | llm.with_structured_output(
        ArticleCard, method="function_calling",
    )
    live_card = card_chain.invoke({"article": ARTICLE})
    print(type(live_card).__name__, live_card.model_dump())
else:
    print("Live Pydantic response skipped.")

Sunday library trial
{'title': 'Sunday library trial', 'summary': 'The library will trial Sunday opening from June for three months.', 'note': None}
Deliberate local failure: ('summary',) string_too_short
Live Pydantic response skipped.


### 4.2 Compare Pydantic, TypedDict, and JSON Schema

Pydantic produces a validated class object. A `TypedDict` declares dictionary fields for
tools and type checkers, but constructing an ordinary dictionary does not enforce those
types at runtime. Raw JSON Schema describes the contract in a language-independent form;
our code must explicitly call a validator when we want local validation of that dictionary.

In the LangChain TypedDict convention below, `Annotated[type, ..., description]` marks a
required field and supplies its description. The ellipsis `...` means no default here.
It is not code omitted from the lesson. We inspect the generated schema locally.

In [9]:
from typing_extensions import TypedDict, Annotated
from langchain_core.utils.function_calling import convert_to_openai_tool
from jsonschema import Draft202012Validator

class ArticleDraft(TypedDict):
    """An article title and summary."""
    title: Annotated[str, ..., "Short factual title"]
    summary: Annotated[str, ..., "One-sentence factual summary"]

# Python still creates this ordinary dictionary.
unchecked: ArticleDraft = {"title": 123}
print("Unchecked dictionary:", unchecked)
tool_definition = convert_to_openai_tool(ArticleDraft)
print("Required fields:",
      tool_definition["function"]["parameters"]["required"])

ARTICLE_SCHEMA = {
    "title": "ArticleCardSchema", "description": "A factual article card",
    "type": "object", "additionalProperties": False,
    "properties": {
        "title": {"type": "string", "minLength": 1, "maxLength": 60},
        "summary": {"type": "string", "minLength": 1, "maxLength": 180},
        "note": {"type": ["string", "null"], "default": None},
    },
    "required": ["title", "summary"],
}
Draft202012Validator.check_schema(ARTICLE_SCHEMA)
dictionary_validator = Draft202012Validator(ARTICLE_SCHEMA)
candidate = dict(VALID_CARD)
dictionary_validator.validate(candidate)
# JSON Schema's default annotation inserts nothing.
assert "note" not in candidate
assert not dictionary_validator.is_valid({**candidate, "note": 42})
assert dictionary_validator.is_valid({**candidate, "note": None})
print("Dictionary passed explicit JSON Schema validation; missing "
      "note stays missing.")

if RUN_LIVE and RUN_EXTRA_LIVE:
    dict_chain = summary_prompt | llm.with_structured_output(
        ArticleDraft, method="function_calling",
    )
    generated_dict = dict_chain.invoke({"article": ARTICLE})
    # Apply the real local checks explicitly before application use.
    dictionary_validator.validate(generated_dict)
    print(generated_dict)

Unchecked dictionary: {'title': 123}
Required fields: ['title', 'summary']
Dictionary passed explicit JSON Schema validation; missing note stays missing.


Pydantic sometimes converts a compatible input before checking it. For a numeric field,
the string `"3"` can become integer `3`. Use strict mode when this conversion would hide an
input mistake. Whether a provider constrains generation is a separate question from which
checks Python performs after receipt. The live examples explicitly select function calling:
the model returns schema-shaped arguments; this does not execute an external tool.

In [10]:
class TrialLength(BaseModel):
    months: int = Field(ge=1, le=12)

print("Converted:", TrialLength.model_validate({"months": "3"}).months)
try:
    TrialLength.model_validate({"months": "3"}, strict=True)
except ValidationError as error:
    print("Strict mode rejected:", error.errors()[0]["type"])

class RequiredNullable(BaseModel):
    note: str | None

assert RequiredNullable.model_validate({"note": None}).note is None
try:
    RequiredNullable.model_validate({})
except ValidationError as error:
    print("Nullable but missing:", error.errors()[0]["type"])

Converted: 3
Strict mode rejected: int_type
Nullable but missing: missing


### 4.3 Allow alternative response structures

If no article is supplied, the model should ask for one instead of inventing a summary.
A **union** allows either of two response types. The outer `final_output` field remains
present in both cases. The `|` in a type annotation means alternatives; it is different
from the `|` connecting Runnables. Pydantic checks the returned structure; it cannot establish
whether the model chose the appropriate response for the user's actual message.

In [11]:
class NeedsArticle(BaseModel):
    message: str = Field(min_length=1)

class SummaryResponse(BaseModel):
    final_output: ArticleCard | NeedsArticle = Field(
        description=
        "ArticleCard when article text is supplied; otherwise NeedsArticle")

local_response = SummaryResponse.model_validate({
    "final_output": {"message": "Please send the article you want summarized."}
})
assert isinstance(local_response.final_output, NeedsArticle)
print(local_response.model_dump())

if RUN_LIVE and RUN_EXTRA_LIVE:
    union_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Summarize supplied article text. If no article was supplied, "
         "ask for it."),
        ("human", "{user_text}"),
    ])
    union_chain = union_prompt | llm.with_structured_output(
        SummaryResponse, method="function_calling",
    )
    response = union_chain.invoke(
        {"user_text": "Can you summarize an article for me?"})
    answer = response.final_output
    print(answer.title if isinstance(answer, ArticleCard) else answer.message)

{'final_output': {'message': 'Please send the article you want summarized.'}}


### 4.4 Display a structured response as it arrives

A parser can yield cumulative partial dictionaries. Replace each displayed field with its
latest value; appending snapshots would repeat earlier text. The local snapshots below are
author-written display test inputs, not model outputs. A nonempty partial summary can pass
field checks before generation finishes, so we wait for stream completion to validate the
final snapshot. Provider errors still propagate; field checks alone cannot prove factual
accuracy or detect every truncated completion.

In [12]:
from IPython.display import clear_output

def display_snapshots(snapshots):
    latest = {}
    for snapshot in snapshots:
        latest = snapshot
        clear_output(wait=True)
        print("Title:", snapshot.get("title", ""))
        print("Summary:", snapshot.get("summary", ""))
    return ArticleCard.model_validate(latest)

test_snapshots = [
    {"title": "Sunday"},
    {"title": "Sunday library trial", "summary": "The library"},
    dict(VALID_CARD),
]
preview_card = display_snapshots(test_snapshots)
print("Final local test snapshot passed field checks.")

if RUN_LIVE and RUN_EXTRA_LIVE:
    preview_chain = summary_prompt | llm.with_structured_output(
        ArticleDraft, method="function_calling",
    )
    final_live_card = display_snapshots(
        preview_chain.stream({"article": ARTICLE}))
    print("Final streamed card:", final_live_card.model_dump())

Title: Sunday library trial
Summary: The library will trial Sunday opening from June for three months.
Final local test snapshot passed field checks.


### 4.5 Guide content with examples; repair validation failures

One input/output demonstration can show the tone and factual detail we want. It is request
context, not weight training. Literal JSON braces inside a format template are doubled;
`{article}` is the actual variable. The example below is author-written, not a measured
quality result.

In [13]:
fewshot_prompt = ChatPromptTemplate.from_messages([
    ("system", 'Write a neutral article card using only supplied facts. '
     'Example article: The museum closes for repairs in July and '
     'reopens in September. '
     'Example output: {{"title": "Museum repairs", '
     '"summary": "The museum closes for repairs in July and '
     'reopens in September."}}'),
    ("human", "{article}"),
])
print(fewshot_prompt.invoke({"article": ARTICLE}).to_messages()[0].content)
if RUN_LIVE and RUN_EXTRA_LIVE:
    example_chain = fewshot_prompt | llm.with_structured_output(
        ArticleCard, method="function_calling",
    )
    print(example_chain.invoke({"article": ARTICLE}).model_dump())

Write a neutral article card using only supplied facts. Example article: The museum closes for repairs in July and reopens in September. Example output: {"title": "Museum repairs", "summary": "The museum closes for repairs in July and reopens in September."}


Validation can identify an empty summary but cannot write its replacement. Our repair
function sends the original article, rejected candidate and actual errors to a replacement
function, then validates again. It permits at most **two replacements**. A valid initial
candidate makes no replacement call. Connection errors are separate from field-validation
failures and are not swallowed here.

The local test uses a deterministic test double returning a known correction, so it tests
our loop without pretending an LLM produced it. The extra live path uses the same function
with an actual model-backed replacement operation.

In [14]:
def validate_and_repair(article, candidate, replacement, max_repairs=2):
    if max_repairs < 0:
        raise ValueError("max_repairs must be nonnegative")
    for attempt in range(max_repairs + 1):
        try:
            return ArticleCard.model_validate(candidate)
        except ValidationError as error:
            if attempt == max_repairs:
                raise
            feedback = str(error)
        candidate = replacement({
            "article": article, "rejected": candidate, "errors": feedback,
        })

repair_feedback = []
def known_local_correction(feedback):
    repair_feedback.append(feedback)
    return dict(VALID_CARD)

invalid_candidate = {"title": "Sunday opening", "summary": ""}
repaired = validate_and_repair(ARTICLE, invalid_candidate,
                               known_local_correction)
assert len(repair_feedback) == 1 and "summary" in repair_feedback[0]["errors"]
assert repair_feedback[0]["article"] == ARTICLE
print("Local test:", len(repair_feedback), "replacement; result:",
      repaired.model_dump())

no_calls = []
validate_and_repair(ARTICLE, VALID_CARD,
                    lambda feedback: no_calls.append(feedback))
assert no_calls == []
failed_calls = []
def still_invalid(feedback):
    failed_calls.append(feedback)
    return dict(invalid_candidate)
try:
    validate_and_repair(ARTICLE, invalid_candidate, still_invalid)
except ValidationError:
    assert len(failed_calls) == 2
    print("Persistent-failure test stopped after two replacements.")

if RUN_LIVE and RUN_EXTRA_LIVE:
    repair_prompt = ChatPromptTemplate.from_messages([
        ("system", "Correct the card using the article and validation errors. "
         "Title: 1-60 characters. Summary: 1-180 characters, one sentence. "
         "Use only the article's facts."),
        ("human",
         "Article: {article}\nRejected: {rejected}\nErrors: {errors}"),
    ])
    repair_chain = repair_prompt | llm.with_structured_output(
        ArticleDraft, method="function_calling",
    )
    print(validate_and_repair(
        ARTICLE, invalid_candidate, repair_chain.invoke,
    ).model_dump())

Local test: 1 replacement; result: {'title': 'Sunday library trial', 'summary': 'The library will trial Sunday opening from June for three months.', 'note': None}
Persistent-failure test stopped after two replacements.


## 5. Extend a chain into an agent workflow
### 5.1 LangGraph and agent loops: state and saved progress

A graph makes execution order explicit. A **node** is a function doing a step; **state**
is the application's working data, not a neural network's hidden state. Nodes return
updates which LangGraph applies to that state. Edges select the next node.

`TypedDict(total=False)` lets a type checker accept omitted keys in this working-state
dictionary. It does not add Python runtime validation; our node functions perform the checks.

This local graph validates an article card, routes an invalid card to a known test
correction, and stops after bounded work. It demonstrates a prescribed repair workflow,
not an autonomous agent or a fake LLM. The test correction stands in for a model call only
so the orchestration can be tested locally.

In [15]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class RepairState(TypedDict, total=False):
    article: str
    candidate: dict
    errors: str
    valid: bool
    repairs: int

def check_card_node(state):
    try:
        ArticleCard.model_validate(state["candidate"])
        return {"valid": True, "errors": ""}
    except ValidationError as error:
        return {"valid": False, "errors": str(error)}

def local_repair_node(state):
    return {
        "candidate": dict(VALID_CARD), "repairs": state.get("repairs", 0) + 1
    }

def after_check(state):
    return "done" if state["valid"] or state.get("repairs",
                                                 0) >= 2 else "repair"

repair_builder = StateGraph(RepairState)
repair_builder.add_node("check", check_card_node)
repair_builder.add_node("repair", local_repair_node)
repair_builder.add_edge(START, "check")
repair_builder.add_conditional_edges("check", after_check,
                                     {"repair": "repair", "done": END})
repair_builder.add_edge("repair", "check")
repair_graph = repair_builder.compile()
graph_result = repair_graph.invoke({
    "article": ARTICLE, "candidate": invalid_candidate, "repairs": 0,
})
assert graph_result["valid"] and graph_result["repairs"] == 1
print("Graph result:", graph_result)

Graph result: {'article': 'The library will open on Sundays from June. The trial lasts three months.', 'candidate': {'title': 'Sunday library trial', 'summary': 'The library will trial Sunday opening from June for three months.'}, 'errors': '', 'valid': True, 'repairs': 1}


An agent differs when its model selects the next tool request after receiving previous
results. The runtime executes an allowed tool and sends its result back to the model.
`create_agent` supplies this model/tool loop using LangGraph.

The optional agent below can only read a tiny local source catalogue. It cannot browse,
send messages, or change files. Its lookup result is evidence to compare with article
claims, not an automatic truth certificate. `ModelCallLimitMiddleware` caps the run at
three model calls. Enabling it can spend API credits.

In [16]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import ModelCallLimitMiddleware

@tool
def lookup_source(source_id: str) -> str:
    """Read one source from the lesson's local source catalogue; valid ID:
    library."""
    catalogue = {
        "library": "Library notice: Sunday opening begins in June; trial "
        "lasts three months."
    }
    return catalogue.get(source_id,
                         "Source not found. Do not invent its contents.")

if RUN_LIVE_AGENT:
    article_agent = create_agent(
        model=llm,
        tools=[lookup_source],
        system_prompt=("Check the article against the local library source "
                       "using lookup_source. Report any contradiction; "
                       "preserve the source's dates and duration."),
        middleware=[
            ModelCallLimitMiddleware(run_limit=3, exit_behavior="error")
        ],
    )
    agent_result = article_agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Check this: The library starts Sunday opening in "
            "June for six months."
        }]
    })
    print(agent_result["messages"][-1].content)
else:
    print("Live agent skipped. Its safe local tool is available, but no agent "
          "model was called.")

Live agent skipped. Its safe local tool is available, but no agent model was called.


Now pause for editorial approval. A **checkpoint** saves state and the execution position;
a **checkpointer** writes and retrieves it. `thread_id` groups one workflow's saved progress.
An **interrupt** asks for external input. On resume, the interrupted node starts again and
the interrupt returns the supplied decision. Our draft node already completed, so it is
not repeated.

We use SQLite in a temporary local directory. The first connection is closed before a new
connection and graph instance resume the saved thread. This tests storage survival across
connections, not a whole-computer crash. The approval resumes in this notebook to keep the
example self-contained; a real application would wait for the editor's request.

In [17]:
import tempfile
from uuid import uuid4
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

class ReviewState(TypedDict, total=False):
    draft: str
    approved: bool
    status: str

draft_calls = []
def prepare_draft(state):
    draft_calls.append("prepared")
    return {"draft": VALID_CARD["summary"]}

def editor_review(state):
    approved = interrupt(
        {"question": "Approve this draft?", "draft": state["draft"]})
    return {
        "approved": bool(approved),
        "status": "approved" if approved else "needs revision"
    }

def review_workflow(checkpointer):
    builder = StateGraph(ReviewState)
    builder.add_node("draft", prepare_draft)
    builder.add_node("review", editor_review)
    builder.add_edge(START, "draft")
    builder.add_edge("draft", "review")
    builder.add_edge("review", END)
    return builder.compile(checkpointer=checkpointer)

checkpoint_directory = tempfile.TemporaryDirectory(
    prefix="lecture5-checkpoints-")
database_path = str(Path(checkpoint_directory.name) / "review.sqlite")
review_config = {"configurable": {"thread_id": str(uuid4())}}

with SqliteSaver.from_conn_string(database_path) as saver:
    paused = review_workflow(saver).invoke({}, config=review_config)
    print("Pause request:", paused["__interrupt__"][0].value)

with SqliteSaver.from_conn_string(database_path) as saver:
    resumed_graph = review_workflow(saver)
    completed = resumed_graph.invoke(Command(resume=True),
                                     config=review_config)
    print("After reopening storage:", completed)

assert completed["approved"] and len(draft_calls) == 1
# Remove only this automatically created temporary test directory.
checkpoint_directory.cleanup()
print("Completed draft node ran once; interrupted review resumed from "
      "its saved thread.")

Pause request: {'question': 'Approve this draft?', 'draft': 'The library will trial Sunday opening from June for three months.'}
After reopening storage: {'draft': 'The library will trial Sunday opening from June for three months.', 'approved': True, 'status': 'approved'}
Completed draft node ran once; interrupted review resumed from its saved thread.


Do not put an unprotected email/payment action before an interrupt: that node restarts on
resume. Place approval before execution and make external operations **idempotent**: repeated
requests with one stable identifier should have the same effect as one request. Checkpoints
do not by themselves guarantee exactly-once side effects. `InMemorySaver` is convenient for
experiments but loses records when its Python process ends.

### 5.2 Middleware: add behavior around agent execution

Middleware is code called by the runtime around agent operations. We can enforce a call
limit without asking the model's permission. This local test uses a clearly scripted model
double that always requests another catalogue lookup. Real LangChain middleware must allow
three calls and block the fourth. The double does no inference and has no network client.

In [18]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain.agents.middleware.model_call_limit import (
    ModelCallLimitExceededError)

class RepeatingToolRequestModel(BaseChatModel):
    """Deterministic test double for exercising the real agent runtime's
    call limit."""
    calls: int = 0

    @property
    def _llm_type(self):
        return "local-scripted-test-double"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.calls += 1
        message = AIMessage(content="", tool_calls=[{
            "name": "lookup_source", "args": {"source_id": "library"},
            "id": f"local-call-{self.calls}", "type": "tool_call",
        }])
        return ChatResult(generations=[ChatGeneration(message=message)])

scripted_model = RepeatingToolRequestModel()
limited_agent = create_agent(
    model=scripted_model, tools=[lookup_source],
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="error")],
)
try:
    limited_agent.invoke({
        "messages": [{"role": "user", "content": "Exercise the call limit."}]
    })
except ModelCallLimitExceededError:
    assert scripted_model.calls == 3
    print(
        "Middleware test passed: three scripted model calls; fourth blocked.")
else:
    raise AssertionError(
        "The configured limit did not stop this deliberately endless test.")

Middleware test passed: three scripted model calls; fourth blocked.


## 6. Inspect, test, and deploy the application
### 6.1 LangSmith: follow a request through its steps

A trace groups recorded operations for a request. A run is one recorded operation, such
as formatting, model invocation, or validation. Our explicit local log below records the
step, duration, and error status. It does not upload content and is not a LangSmith trace.
`@contextmanager` lets our function surround a `with` block: `yield` marks where that block
runs, between the function's setup and cleanup code.
If a summary says six months, recorded prompt and model-output data can help establish
where the change appeared; timing data alone cannot establish factual correctness.

Custom application functions need explicit instrumentation. The optional hosted example makes
one additional model request when enabled. Hosted tracing is opt-in:
review what input/output data will leave your machine before enabling it. Do not log keys.

In [19]:
from contextlib import contextmanager

local_runs = []
@contextmanager
def record_local_step(name):
    started = time.perf_counter()
    record = {"step": name, "error": False}
    try:
        yield
    except Exception:
        record["error"] = True
        raise
    finally:
        record["seconds"] = time.perf_counter() - started
        local_runs.append(record)

with record_local_step("validate_article_card"):
    ArticleCard.model_validate(VALID_CARD)
print(local_runs)

# Optional hosted example: remains disabled unless BOTH switches and a
# LangSmith key are set.
if ENABLE_LANGSMITH and RUN_LIVE:
    if not os.getenv("LANGSMITH_API_KEY"):
        raise RuntimeError(
            "Set LANGSMITH_API_KEY privately before opting "
            "into hosted tracing.")
    from langsmith import traceable, tracing_context

    @traceable(name="lecture5_summary")
    def traced_summary(article):
        return summary_chain.invoke({"article": article})

    # Only this public lesson article is submitted in this demonstration.
    with tracing_context(enabled=True, project_name="lecture5-public-demo"):
        print(traced_summary(ARTICLE))
else:
    print("Hosted tracing disabled; local log stays in this kernel.")

[{'step': 'validate_article_card', 'error': False, 'seconds': 8.832983439788222e-06}]
Hosted tracing disabled; local log stays in this kernel.


### 6.2 Test datasets, evaluators, and human feedback

Save inputs separately from reference requirements. The summarizer receives only the article;
the evaluator receives its answer and the reference requirements. For the library article,
a correct summary retains Sunday opening, June and three months without contradiction.
Field validation alone does not establish that.

The following tiny evaluator tests three author-written candidate answers. It checks exact
phrases to make the mechanism visible; it is suitable only for this controlled regression
test, not arbitrary paraphrases. A real evaluation needs reviewed cases, semantic checks
where appropriate, and checks of the judge's decisions against human judgments.

In [20]:
REFERENCE_REQUIREMENTS = ("sunday", "june", "three months")
evaluation_cases = [
    {"name": "complete", "output": dict(VALID_CARD)},
    {
        "name": "duration_changed", "output": {
            "title": "Sunday trial",
            "summary": "The library opens Sundays from June for six months."
        }
    },
    {
        "name": "duration_omitted", "output": {
            "title": "Sunday trial",
            "summary": "The library opens Sundays from June."
        }
    },
]

def evaluate_controlled_card(candidate):
    try:
        card = ArticleCard.model_validate(candidate)
    except ValidationError:
        return {"fields_ok": False, "facts_ok": False, "passed": False}
    text = card.summary.casefold()
    facts_ok = all(
        fact in text
        for fact in REFERENCE_REQUIREMENTS) and "six months" not in text
    return {"fields_ok": True, "facts_ok": facts_ok, "passed": facts_ok}

results = [{"case": case["name"], **evaluate_controlled_card(case["output"])}
           for case in evaluation_cases]
for result in results:
    print(result)
passed = sum(result["passed"] for result in results)
print(f"Passed both field and content checks: {passed}/{len(results)} "
      f"author-written cases")
assert passed == 1

{'case': 'complete', 'fields_ok': True, 'facts_ok': True, 'passed': True}
{'case': 'duration_changed', 'fields_ok': True, 'facts_ok': False, 'passed': False}
{'case': 'duration_omitted', 'fields_ok': True, 'facts_ok': False, 'passed': False}
Passed both field and content checks: 1/3 author-written cases


To compare prompt A and B, run both versions on the same reviewed articles with the same
model, generation settings, and checks. Save each version's outputs and measured results as
an experiment; keep some cases out of prompt development. Repeat stochastic comparisons
when one lucky generation could change the decision. A LangSmith dataset stores examples,
experiments store tested configurations/results, and annotation queues organize human review.
An editor's correction can improve references or a judge's instructions; it does not
automatically train the summarizer's weights.

### 6.3 Prompt versions, production monitoring, and costs

A release should identify its prompt revision, model ID, settings, schema, workflow code,
and evaluation-dataset version. A prompt version alone does not reconstruct the application.
On real requests, monitor errors, response times, tokens and extra calls, then inspect sampled
content for factual failures. A request can finish without an exception yet contain a wrong
duration. Token-based cost estimates require the actual model/provider's current rates and
may omit gateway fees or tool charges.

In [21]:
import hashlib
import json

prompt_text = summary_prompt.to_json()
release = {
    "application": "lecture5-article-assistant",
    "prompt_sha256": hashlib.sha256(
        json.dumps(prompt_text, sort_keys=True, default=str).encode()
    ).hexdigest(),
    "model": MODEL_ID,
    "temperature": 0,
    "schema": "ArticleCard-v1",
    "workflow": "lesson5-v1",
    "evaluation_dataset": "controlled-library-regressions-v1",
}
print(json.dumps(release, indent=2))

# Controlled operations log, not an observed production report.
request_log = [
    {"error": False, "seconds": 0.5},
    {"error": True, "seconds": 0.8},
    {"error": False, "seconds": 0.4},
]
print("Errors / all requests:", sum(r["error"] for r in request_log), "/",
      len(request_log))
print("Mean seconds:",
      sum(r["seconds"] for r in request_log) / len(request_log))

def estimate_model_cost(input_tokens, output_tokens, input_usd_per_million,
                        output_usd_per_million):
    return (input_tokens * input_usd_per_million +
            output_tokens * output_usd_per_million) / 1_000_000

print("Cost helper defined; supply verified current rates rather than "
      "stale example prices.")

{
  "application": "lecture5-article-assistant",
  "prompt_sha256": "bf11b299b319af23ba242605e9ed976fc22a056d3deddf1d095d7bd02f60ec51",
  "model": "openai/gpt-4o-mini",
  "temperature": 0,
  "schema": "ArticleCard-v1",
  "workflow": "lesson5-v1",
  "evaluation_dataset": "controlled-library-regressions-v1"
}
Errors / all requests: 1 / 3
Mean seconds: 0.5666666666666668
Cost helper defined; supply verified current rates rather than stale example prices.


### 6.4 Deploy an application so other people can use it

Deployment packages application code, dependencies and private configuration into a running
service. A website sends the article to that service, the Python workflow calls the model
through OpenRouter, and the service returns the answer. The model remains at its provider.

An ordinary Python web service can host LangChain/LangGraph without LangSmith. LangSmith
Deployment is an optional commercial hosting product; its Agent Server serves graph APIs.
Studio is a developer interface, not the user's website. `RemoteGraph` is a client that
invokes a graph on an already-running remote server. This notebook does not deploy anything.

Before serving other people, implement authentication, per-user state access, protected
credentials, input-size limits, request/time/spending limits, durable checkpoint storage,
error handling and tested permissions for every tool. A starter template provides files,
not your application's safety policy.

## References and next steps

- [LangChain overview](https://docs.langchain.com/oss/python/langchain/overview)
- [OpenRouter integration](https://docs.langchain.com/oss/python/integrations/chat/openrouter)
- [RunnableSequence](https://reference.langchain.com/python/langchain-core/runnables/base/RunnableSequence)
- [Pydantic fields](https://docs.pydantic.dev/latest/concepts/fields/)
- [JSON Schema annotations](https://json-schema.org/understanding-json-schema/reference/annotations)
- [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangChain middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview)
- [LangSmith evaluation](https://docs.langchain.com/langsmith/evaluation-concepts)
- [LangSmith deployment](https://docs.langchain.com/langsmith/deployment)

Run all cells first with live switches off. Then enable the three core model requests.
Inspect their real outputs before enabling additional demonstrations. Change one setting
at a time so you can connect the change to what the application actually did.